In [ ]:
# # ==== SETUP: Download data files ====
# # Run this cell ONCE to download the data folder from Google Drive.
# # After it finishes, you can skip this cell in future runs.

# import subprocess, sys, os
# subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])
# import gdown
# GOOGLE_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1ei8-m-N2vJHj2y7LHA7QYEC7FT2G-qDN?usp=sharing"

# output_dir = "data"
# os.makedirs(output_dir, exist_ok=True)

# gdown.download_folder(GOOGLE_DRIVE_FOLDER_URL, output=output_dir, quiet=False)

# print("Done! All data files downloaded to the 'data/' folder.")

In [ ]:
import pandas as pd
#1: data colleciton
# Load all 4 files
ff_factors = pd.read_csv("data/F-F_Research_Data_Factors.csv", skiprows=3)
ff_mom = pd.read_csv("data/F-F_Momentum_Factor.csv", skiprows=13)
sic_mapping = pd.read_excel("data/SIC_49_Industry.xlsx")
oap_factors = pd.read_csv("data/signed_predictors_dl_wide.csv")
crsp = pd.read_csv("data/crsp.csv")

# Print heads
print("=== FF 3 Factors ===")
print(ff_factors.head())
print(f"\nShape: {ff_factors.shape}")

print("\n=== FF Momentum ===")
print(ff_mom.head())
print(f"\nShape: {ff_mom.shape}")

print("\n=== SIC 49 Industry Mapping ===")
print(sic_mapping.head())
print(f"\nShape: {sic_mapping.shape}")

print("\n=== OAP Factors ===")
print(oap_factors.head())
print(f"\nShape: {oap_factors.shape}")

print("\n=== CRSP ===")
print(crsp.head())
print(f"\nShape: {crsp.shape}")

/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_97020/4241882157.py:8: DtypeWarning: Columns (0: SICCD) have mixed types. Specify dtype option on import or set low_memory=False.
  crsp = pd.read_csv("data/crsp.csv")


=== FF 3 Factors ===
  Unnamed: 0   Mkt-RF      SMB      HML       RF
0     192607     2.89    -2.55    -2.39     0.22
1     192608     2.64    -1.14     3.81     0.25
2     192609     0.38    -1.36     0.05     0.23
3     192610    -3.27    -0.14     0.82     0.32
4     192611     2.54    -0.11    -0.61     0.31

Shape: (1298, 5)

=== FF Momentum ===
  Unnamed: 0      Mom
0     192701     0.57
1     192702    -1.50
2     192703     3.52
3     192704     4.36
4     192705     2.78

Shape: (1293, 2)

=== SIC 49 Industry Mapping ===
   Industry  SIC_start  SIC_end Industry_name
0         1        100      199         Agric
1         1        200      299         Agric
2         1        700      799         Agric
3         1        910      919         Agric
4         1       2048     2048         Agric

Shape: (598, 4)

=== OAP Factors ===
   permno  yyyymm  AM  AOP  AbnormalAccruals  Accruals  AccrualsBM  Activism1  \
0   10000  198601 NaN  NaN               NaN       NaN         NaN  

In [ ]:
#2: Data cleaning
import pandas as pd
import numpy as np

#config the 5 and 100 thresholds are given per assignemnt
DATA_DIR = "data"
OUTPUT_DIR = "cleaned_data"
START_YEAR = 1985
END_YEAR = 2023
MIN_PRICE = 5.0
MIN_MARKET_CAP = 500  # raised from 100M to 500M to reduce microcap tilt

#CRSP cleaning:
before_rows = len(crsp)
crsp.columns = crsp.columns.str.upper()
crsp['DATE'] = pd.to_datetime(crsp['DATE'])
crsp['YEAR'] = crsp['DATE'].dt.year
crsp['MONTH'] = crsp['DATE'].dt.month
crsp['YYYYMM'] = crsp['YEAR'] * 100 + crsp['MONTH']
crsp = crsp[crsp['SHRCD'].isin([10, 11])] 
crsp = crsp[crsp['EXCHCD'].isin([1, 2, 3])]
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp = crsp[crsp['RET'].notna()]
crsp = crsp[crsp['RET'] > -1.0] 
crsp['PRC_ABS'] = crsp['PRC'].abs()
crsp['MARKET_CAP'] = crsp['PRC_ABS'] * crsp['SHROUT'] 
crsp = crsp[crsp['PRC_ABS'] >= MIN_PRICE] 
crsp = crsp[crsp['MARKET_CAP'] >= MIN_MARKET_CAP * 1000] 
crsp = crsp[(crsp['YEAR'] >= START_YEAR) & (crsp['YEAR'] <= END_YEAR)]
 
print(f"CRSP cleaned: {len(crsp):,} rows. (Removed {before_rows - len(crsp):,} rows)")


In [ ]:
# Vectorized FF49 mapping: interval lookup instead of per-row Python loop
sic_mapping_sorted = sic_mapping.sort_values('SIC_start').reset_index(drop=True)
intervals = pd.IntervalIndex.from_arrays(
    sic_mapping_sorted['SIC_start'],
    sic_mapping_sorted['SIC_end'],
    closed='both',
)
industry_values = sic_mapping_sorted['Industry'].to_numpy()

sic_numeric = pd.to_numeric(crsp['SICCD'], errors='coerce')
idx = intervals.get_indexer(sic_numeric.fillna(-1).to_numpy())
crsp['FF49'] = np.where(idx >= 0, industry_values[idx], 49).astype('int16')

# Vectorized annual compounding via log1p/expm1 + groupby.sum (C-level, no Python apply)
log1p_ret = np.log1p(crsp['RET'].to_numpy())
annual_returns = (
    pd.DataFrame({
        'PERMNO': crsp['PERMNO'].to_numpy(),
        'YEAR': crsp['YEAR'].to_numpy(),
        'LOG1P_RET': log1p_ret,
    })
    .groupby(['PERMNO', 'YEAR'], sort=False, as_index=False)['LOG1P_RET']
    .sum()
)
annual_returns['RET_ANNUAL'] = np.expm1(annual_returns['LOG1P_RET'])
annual_returns = annual_returns[['PERMNO', 'YEAR', 'RET_ANNUAL']]

print(f"Annual returns: {len(annual_returns):,} stock-years")

oap_rows_before = len(oap_factors)
dec_crsp = crsp.loc[crsp['MONTH'] == 12, ['PERMNO', 'YEAR', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']].copy()
oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
oap_factors['MONTH'] = oap_factors['yyyymm'] % 100
oap_factors_dec = oap_factors[oap_factors['MONTH'] == 12].copy()
oap_factors_dec['RETURN_YEAR'] = oap_factors_dec['YEAR'] + 1
print(f"oap_factors December: {len(oap_factors_dec):,} rows. Before: {oap_rows_before:,} rows.")


Annual returns: 124,875 stock-years


/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_97020/3489364595.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_97020/3489364595.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oap_factors['MONTH'] = oap_factors['yyyymm'] % 100


oap_factors December: 454,907 rows. Before: 5,416,424 rows.


In [ ]:
oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
oap_factors['MONTH'] = oap_factors['yyyymm'] % 100
oap_factors_dec = oap_factors[oap_factors['MONTH'] == 12].copy()
oap_factors_dec['RETURN_YEAR'] = oap_factors_dec['YEAR'] + 1  # Dec T factors predict T+1 returns

print(f"OAP December: {len(oap_factors_dec):,} rows")

# LEFT join (was inner) so stocks delisting in T+1 aren't silently dropped.
# Survivorship-bias correction: impute -30% as the assumed delisting return
# (Shumway 1997 convention) for stocks with no T+1 returns in cleaned CRSP.
DELISTING_RETURN = -0.30

merged = pd.merge(
    oap_factors_dec,
    annual_returns,
    left_on=['permno', 'RETURN_YEAR'],
    right_on=['PERMNO', 'YEAR'],
    how='left'
)
n_delisted = int(merged['RET_ANNUAL'].isna().sum())
merged['RET_ANNUAL'] = merged['RET_ANNUAL'].fillna(DELISTING_RETURN)
print(f"Survivorship correction: imputed {n_delisted:,} stock-years to {DELISTING_RETURN:.0%}")

# Merge with December CRSP data for FF49 / market cap
merged = pd.merge(
    merged,
    dec_crsp,
    left_on=['permno', 'YEAR_x'],
    right_on=['PERMNO', 'YEAR'],
    how='left'
)
merged.head()


In [ ]:
ff_factors.columns = ['YYYYMM', 'MKT_RF', 'SMB', 'HML', 'RF']
ff_factors = ff_factors[ff_factors['YYYYMM'].astype(str).str.len() == 6]
ff_factors['YYYYMM'] = ff_factors['YYYYMM'].astype(int)
ff_factors[['MKT_RF', 'SMB', 'HML', 'RF']] = ff_factors[['MKT_RF', 'SMB', 'HML', 'RF']].apply(pd.to_numeric, errors='coerce') / 100
 
ff_mom.columns = ['YYYYMM', 'UMD']
ff_mom = ff_mom[ff_mom['YYYYMM'].astype(str).str.len() == 6]
ff_mom['YYYYMM'] = ff_mom['YYYYMM'].astype(int)
ff_mom['UMD'] = pd.to_numeric(ff_mom['UMD'], errors='coerce') / 100
 
ff_all = pd.merge(ff_factors, ff_mom, on='YYYYMM', how='inner')
ff_all.head()

,YYYYMM,MKT_RF,SMB,HML,RF,UMD
0,192701,-0.0005,-0.0032,0.0458,0.0025,0.0057
1,192702,0.0417,0.0007,0.0272,0.0026,-0.0150
2,192703,0.0014,-0.0177,-0.0238,0.0030,0.0352
3,192704,0.0047,0.0039,0.0065,0.0025,0.0436
4,192705,0.0545,0.0155,0.0480,0.0030,0.0278


In [ ]:
id_cols = ['permno', 'yyyymm', 'YEAR', 'MONTH', 'RETURN_YEAR', 'PERMNO', 'YEAR_x', 'YEAR_y',
           'RET_ANNUAL', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']
factor_cols = [c for c in merged.columns if c not in id_cols]
 
missing_pct = (merged[factor_cols].isna().sum() / len(merged) * 100).sort_values()
print("Missing data percentage per factor:")
print(missing_pct)

Missing data percentage per factor:
PERMNO_x                 0.000000
DivInit                  0.032327
DivOmit                  0.032327
MaxRet                   0.213530
ExchSwitch               0.239051
                          ...    
AccrualsBM              95.106680
Recomm_ShortInterest    96.615851
Activism2               98.102903
ProbInformedTrading     99.029333
IO_ShortInterest        99.196073
Length: 211, dtype: float64


In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged = merged[merged['FF49'].notna()].copy()

def _downcast(df):
    out = df.copy()
    f64 = out.select_dtypes(include='float64').columns
    i64 = out.select_dtypes(include='int64').columns
    out[f64] = out[f64].astype('float32')
    out[i64] = out[i64].astype('int32')
    return out

_downcast(merged).to_parquet(f"{OUTPUT_DIR}/cleaned_merged_data.parquet", index=False, compression='zstd', compression_level=19)
ff_all.to_parquet(f"{OUTPUT_DIR}/ff_factors_clean.parquet", index=False, compression='zstd', compression_level=19)
print(f"Wrote {OUTPUT_DIR}/cleaned_merged_data.parquet, ff_factors_clean.parquet")


In [ ]:
print(merged.head())
print(ff_all.head())

    permno  yyyymm        AM       AOP  AbnormalAccruals  Accruals  \
2    10001  201412  1.762830  0.042705         -0.005713  0.034970   
4    10001  201612  1.497349       NaN         -0.039445  0.063455   
6    10002  199712  3.365251       NaN               NaN -0.010779   
7    10002  199812  3.135689       NaN               NaN  0.033185   
11   10002  200312  4.752564       NaN               NaN -0.011305   

    AccrualsBM  Activism1  Activism2     AdExp  ...  RETURN_YEAR  PERMNO_x  \
2          NaN        NaN        NaN       NaN  ...         2015     10001   
4          NaN        NaN        NaN       NaN  ...         2017     10001   
6          NaN        NaN        NaN  0.001250  ...         1998     10002   
7          NaN        NaN        NaN  0.001425  ...         1999     10002   
11         NaN        NaN        NaN  0.002777  ...         2004     10002   

    YEAR_y  RET_ANNUAL  PERMNO_y    YEAR  PRC_ABS  MARKET_CAP   SICCD  FF49  
2     2015   -0.065084   10001.0

In [ ]:
print("Data cleaning complete. ")

Data cleaning complete. 


In [ ]:
#3: Z score normalization
print(merged.columns.to_list())
id_cols = ['permno', 'yyyymm', 'YEAR_x', 'MONTH', 'RETURN_YEAR', 'PERMNO_x', 
           'YEAR_y', 'RET_ANNUAL', 'PERMNO_y', 'YEAR', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']
factor_cols = [c for c in merged.columns if c not in id_cols]
print(f"Factors to z-score: {len(factor_cols)}")


['permno', 'yyyymm', 'AM', 'AOP', 'AbnormalAccruals', 'Accruals', 'AccrualsBM', 'Activism1', 'Activism2', 'AdExp', 'AgeIPO', 'AnalystRevision', 'AnalystValue', 'AnnouncementReturn', 'AssetGrowth', 'BM', 'BMdec', 'BPEBM', 'Beta', 'BetaFP', 'BetaLiquidityPS', 'BetaTailRisk', 'BidAskSpread', 'BookLeverage', 'BrandInvest', 'CBOperProf', 'CF', 'CPVolSpread', 'Cash', 'CashProd', 'ChAssetTurnover', 'ChEQ', 'ChForecastAccrual', 'ChInv', 'ChInvIA', 'ChNAnalyst', 'ChNNCOA', 'ChNWC', 'ChTax', 'ChangeInRecommendation', 'CitationsRD', 'CompEquIss', 'CompositeDebtIssuance', 'ConsRecomm', 'ConvDebt', 'CoskewACX', 'Coskewness', 'CredRatDG', 'CustomerMomentum', 'DebtIssuance', 'DelBreadth', 'DelCOA', 'DelCOL', 'DelDRC', 'DelEqu', 'DelFINL', 'DelLTI', 'DelNetFin', 'DivInit', 'DivOmit', 'DivSeason', 'DivYieldST', 'DolVol', 'DownRecomm', 'EBM', 'EP', 'EarnSupBig', 'EarningsConsistency', 'EarningsForecastDisparity', 'EarningsStreak', 'EarningsSurprise', 'EntMult', 'EquityDuration', 'ExchSwitch', 'ExclExp',

In [ ]:
merged_z = merged.copy()

grouped = merged_z.groupby(['FF49', 'RETURN_YEAR'])[factor_cols]
means = grouped.transform('mean')
stds = grouped.transform('std')

merged_z[factor_cols] = (merged_z[factor_cols] - means) / stds.replace(0, np.nan)
merged_z[factor_cols] = merged_z[factor_cols].fillna(0)

# IS-only winsor cutoffs (1986-2007), applied to all years -> no OOS look-ahead.
WINSOR_LO, WINSOR_HI = 0.01, 0.99
IS_START, IS_END = 1986, 2007
is_panel = merged_z[(merged_z['RETURN_YEAR'] >= IS_START) & (merged_z['RETURN_YEAR'] <= IS_END)]
print(f"Fitting winsor on IS ({IS_START}-{IS_END}, {len(is_panel):,} rows)")

lower_q = is_panel[factor_cols].quantile(WINSOR_LO)
upper_q = is_panel[factor_cols].quantile(WINSOR_HI)
merged_z[factor_cols] = merged_z[factor_cols].clip(lower=lower_q, upper=upper_q, axis=1)
print(f"Z-scored factors winsorized at IS [{WINSOR_LO:.0%}, {WINSOR_HI:.0%}]")

ret_lo = float(is_panel['RET_ANNUAL'].quantile(WINSOR_LO))
ret_hi = float(is_panel['RET_ANNUAL'].quantile(WINSOR_HI))
n_clipped = int(((merged_z['RET_ANNUAL'] < ret_lo) | (merged_z['RET_ANNUAL'] > ret_hi)).sum())
merged_z['RET_ANNUAL'] = merged_z['RET_ANNUAL'].clip(lower=ret_lo, upper=ret_hi)
print(f"RET_ANNUAL winsorized at IS [{ret_lo:.3f}, {ret_hi:.3f}] ({n_clipped:,} clipped)")

print("Z-scoring + winsorization complete")


In [ ]:
print(f"Sample z-scored factor mean: {merged_z['BM'].mean():.4f} (should be ~0)")
print(f"Sample z-scored factor std:  {merged_z['BM'].std():.4f} (after winsorization, < 1)")
_downcast(merged_z).to_parquet(f"{OUTPUT_DIR}/cleaned_merged_zscored.parquet", index=False, compression='zstd', compression_level=19)
merged_z.head()


In [ ]:
#4: Fama Macbeth
IN_SAMPLE_START = 1986
IN_SAMPLE_END = 2007

in_sample = merged_z[(merged_z['RETURN_YEAR'] >= IN_SAMPLE_START) & 
                      (merged_z['RETURN_YEAR'] <= IN_SAMPLE_END)]

print(f"In-sample: {len(in_sample):,} rows")
print(f"Years: {in_sample['RETURN_YEAR'].min()} - {in_sample['RETURN_YEAR'].max()}")
print(f"Unique years: {in_sample['RETURN_YEAR'].nunique()}")


In-sample: 59,181 rows
Years: 1986 - 2007
Unique years: 22


In [ ]:
#4: Two-stage Fama-MacBeth on the full 209-factor OAP universe
# Stage 1 univariate -> keep top 40 by |t| -> Stage 2 multivariate on those 40

import statsmodels.api as sm
from collections import defaultdict

# Use ALL 209 factor columns (defined earlier as factor_cols)
print(f"Full factor universe: {len(factor_cols)} factors")

UNIVARIATE_TOP_K = 40

# ---------- Stage 1: Univariate FM ----------
print(f"\nStage 1: Univariate FM across {len(factor_cols)} factors...")
uni_yearly = defaultdict(list)
for year in sorted(in_sample['RETURN_YEAR'].unique()):
    yd = in_sample[in_sample['RETURN_YEAR'] == year]
    if len(yd) < 50:
        continue
    y = yd['RET_ANNUAL'].to_numpy()
    for f in factor_cols:
        x = yd[f].to_numpy()
        X = sm.add_constant(x)
        try:
            coef = float(sm.OLS(y, X).fit().params[1])
        except Exception:
            coef = np.nan
        uni_yearly[f].append(coef)

def _aggregate(yearly_dict):
    rows = []
    for f, coefs in yearly_dict.items():
        clean = [c for c in coefs if not np.isnan(c)]
        if len(clean) < 5:
            continue
        avg = np.mean(clean)
        std = np.std(clean, ddof=1)
        t = avg / (std/np.sqrt(len(clean))) if std > 0 else 0.0
        rows.append({'Factor': f, 'Avg_Premium': avg, 'Std_Premium': std,
                     't_stat': t, 'N_Years': len(clean)})
    df = pd.DataFrame(rows)
    df['abs_t'] = df['t_stat'].abs()
    return df.sort_values('abs_t', ascending=False).reset_index(drop=True)

uni_df = _aggregate(uni_yearly)
top_k_factors = uni_df.head(UNIVARIATE_TOP_K)['Factor'].tolist()
print(f"\nTop {UNIVARIATE_TOP_K} by univariate |t|:")
print(uni_df.head(UNIVARIATE_TOP_K)[['Factor','Avg_Premium','t_stat']].to_string(index=False))

# ---------- Stage 2: Multivariate FM on top 40 ----------
print(f"\nStage 2: Multivariate FM on top {len(top_k_factors)} factors jointly...")
multi_yearly = defaultdict(list)
for year in sorted(in_sample['RETURN_YEAR'].unique()):
    yd = in_sample[in_sample['RETURN_YEAR'] == year]
    clean = yd[top_k_factors + ['RET_ANNUAL']].dropna()
    if len(clean) < 50:
        continue
    X = sm.add_constant(clean[top_k_factors])
    y = clean['RET_ANNUAL']
    try:
        model = sm.OLS(y, X).fit()
    except Exception:
        continue
    for f in top_k_factors:
        multi_yearly[f].append(float(model.params.get(f, np.nan)))
    print(f"  Year {year}: {len(clean):,} obs")


In [ ]:
# Aggregate multivariate FM into final fm_df
fm_df = _aggregate(multi_yearly)
print(f"\nMultivariate FM result ({len(fm_df)} factors):")
print(fm_df[['Factor','Avg_Premium','t_stat','N_Years']].to_string(index=False))


In [ ]:
print("ALL MULTIVARIATE FACTORS BY |t-stat|:\n")
print(fm_df[['Factor','Avg_Premium','t_stat','N_Years']].to_string(index=False))


In [ ]:
#5: select final model — top N multivariate factors by |t|, sign-agnostic.
# Negative-premium factors get negative t_stat weights -> shorted in composite SCORE.

TOP_N = 25
final_view = fm_df.sort_values('abs_t', ascending=False).head(TOP_N).copy()
n_neg = int((final_view['t_stat'] < 0).sum())
print(f"Top {TOP_N} by |t| (no sign filter): {n_neg} negative-premium factors\n")
print(final_view[['Factor','Avg_Premium','t_stat','N_Years']].to_string(index=False))

# Keep variable name `sign_ok` for the next cell so we don't have to rename downstream.
sign_ok = final_view


In [ ]:
final_factors = sign_ok[['Factor', 't_stat']].reset_index(drop=True)
final_factors.to_parquet(f"{OUTPUT_DIR}/final_factors.parquet", index=False, compression='zstd', compression_level=19)
print(f"Saved {len(final_factors)} sign-checked factors as final model")


In [ ]:
#Out of Sample Scoring:
OOS_START = 2008
OOS_END = 2023

oos_data = merged_z[(merged_z['RETURN_YEAR'] >= OOS_START) & 
                     (merged_z['RETURN_YEAR'] <= OOS_END)]

print(f"Out-of-sample: {len(oos_data):,} rows")
print(f"Years: {oos_data['RETURN_YEAR'].min()} - {oos_data['RETURN_YEAR'].max()}")


Out-of-sample: 42,885 rows
Years: 2008 - 2023


In [ ]:
final_factors = pd.read_parquet(f"{OUTPUT_DIR}/final_factors.parquet")
factor_names = final_factors['Factor'].tolist()
factor_weights = dict(zip(final_factors['Factor'], final_factors['t_stat']))

print(f"Scoring with {len(factor_names)} factors")
oos_data = oos_data.copy()
valid_factors = [f for f in factor_names if f in oos_data.columns]
print(f"Valid factors in data: {len(valid_factors)}")
oos_data['SCORE'] = 0
for factor in valid_factors:
    weight = factor_weights[factor]
    oos_data['SCORE'] += oos_data[factor].fillna(0) * weight

print(f"Scored {len(oos_data):,} stock-years")
print(f"Score range: {oos_data['SCORE'].min():.2f} to {oos_data['SCORE'].max():.2f}")


In [ ]:
oos_data[['permno', 'RETURN_YEAR', 'SCORE', 'RET_ANNUAL']].head(10)

,permno,RETURN_YEAR,SCORE,RET_ANNUAL
2,10001,2015,-8.035746,-0.065084
4,10001,2017,8.043116,0.043988
15,10002,2008,0.193764,0.282612
16,10002,2009,6.487220,-0.552441
64,10025,2008,-3.709663,-0.450797
65,10025,2009,2.068374,1.933332
66,10025,2010,-7.176062,-0.322099
67,10025,2011,8.091149,0.084778
68,10025,2012,12.045550,1.104086
69,10025,2013,8.688757,-0.108052


In [ ]:
#Build Portfolio — explicit decile sort (D10 = top, D1 = bottom) per OOS year
N_BUCKETS = 10

def assign_deciles(group):
    try:
        return pd.qcut(group['SCORE'], N_BUCKETS, labels=False, duplicates='drop') + 1
    except ValueError:
        return pd.Series(np.nan, index=group.index)

oos_data = oos_data.copy()
oos_data['decile'] = (
    oos_data.groupby('RETURN_YEAR', group_keys=False).apply(assign_deciles)
)
oos_data = oos_data.dropna(subset=['decile'])
oos_data['decile'] = oos_data['decile'].astype(int)

portfolio_returns = []
for year in sorted(oos_data['RETURN_YEAR'].unique()):
    yd = oos_data[oos_data['RETURN_YEAR'] == year]
    long_ret = yd.loc[yd['decile'] == N_BUCKETS, 'RET_ANNUAL'].mean()
    short_ret = yd.loc[yd['decile'] == 1, 'RET_ANNUAL'].mean()
    portfolio_returns.append({
        'Year': year,
        'Long_Ret': long_ret,
        'Short_Ret': short_ret,
        'Spread': long_ret - short_ret,
        'N_Long': int((yd['decile'] == N_BUCKETS).sum()),
        'N_Short': int((yd['decile'] == 1).sum()),
    })

port_df = pd.DataFrame(portfolio_returns)
print(port_df.to_string(index=False))
print(f"\n=== ANNUAL D10-D1 PERFORMANCE (OOS {OOS_START}-{OOS_END}) ===")
print(f"Avg Long  (D10) Return: {port_df['Long_Ret'].mean()*100:.2f}%")
print(f"Avg Short (D1)  Return: {port_df['Short_Ret'].mean()*100:.2f}%")
print(f"Avg L-S Spread:         {port_df['Spread'].mean()*100:.2f}%")
print(f"Spread Std Dev:         {port_df['Spread'].std()*100:.2f}%")
print(f"Spread t-stat:          {port_df['Spread'].mean() / (port_df['Spread'].std() / np.sqrt(len(port_df))):.2f}")


 Year  Long_Ret  Short_Ret    Spread  N_Long  N_Short
 2008 -0.146872  -0.308614  0.161743     306      306
 2009  0.660739   0.402130  0.258609     222      222
 2010  0.350047   0.242188  0.107859     250      250
 2011  0.029083  -0.022670  0.051753     264      264
 2012  0.245258   0.184731  0.060527     244      244
 2013  0.510241   0.392260  0.117981     250      250
 2014  0.116192   0.049369  0.066823     273      273
 2015  0.024018  -0.043666  0.067684     277      277
 2016  0.322206   0.201978  0.120228     268      268
 2017  0.336258   0.148735  0.187524     265      266
 2018  0.014773  -0.083904  0.098677     267      267
 2019  0.280276   0.327742 -0.047466     257      257
 2020  0.500430   0.288560  0.211870     260      261
 2021  0.178423   0.183628 -0.005206     287      287
 2022 -0.035604  -0.156632  0.121028     326      326
 2023  0.285910   0.256711  0.029199     277      277

=== ANNUAL D10-D1 PERFORMANCE (OOS 2008-2023) ===
Avg Long  (D10) Return: 22.95%


In [ ]:
# Monthly value-weighted L/S series (Dec-T MARKET_CAP weights, fixed for the year)
decile_map = oos_data[['permno', 'RETURN_YEAR', 'decile', 'MARKET_CAP']]
oos_years = oos_data['RETURN_YEAR'].unique()
monthly = crsp.loc[crsp['YEAR'].isin(oos_years), ['PERMNO', 'YEAR', 'YYYYMM', 'RET']].copy()
joined = monthly.merge(
    decile_map,
    left_on=['PERMNO', 'YEAR'],
    right_on=['permno', 'RETURN_YEAR'],
    how='inner',
)
joined = joined.dropna(subset=['RET', 'MARKET_CAP'])
joined['W_RET'] = joined['RET'] * joined['MARKET_CAP']

grp = joined.groupby(['YYYYMM', 'decile'])
monthly_dec = (grp['W_RET'].sum() / grp['MARKET_CAP'].sum()).unstack('decile').sort_index()
monthly_dec.columns = [f'D{int(c)}' for c in monthly_dec.columns]

monthly_df = monthly_dec.copy()
monthly_df['Long_Ret'] = monthly_df[f'D{N_BUCKETS}']
monthly_df['Short_Ret'] = monthly_df['D1']
monthly_df['Spread'] = monthly_df['Long_Ret'] - monthly_df['Short_Ret']
monthly_df['Cumulative'] = (1.0 + monthly_df['Spread']).cumprod() - 1.0
monthly_df = monthly_df.reset_index()

print(f"Value-weighted monthly returns: {len(monthly_df)} months ({monthly_df['YYYYMM'].min()}-{monthly_df['YYYYMM'].max()})")
print(f"Cumulative VW D10-D1 return: {monthly_df['Cumulative'].iloc[-1]*100:.2f}%")


In [ ]:
ff_all = pd.read_parquet(f"{OUTPUT_DIR}/ff_factors_clean.parquet")
monthly_df = pd.merge(monthly_df, ff_all, on='YYYYMM', how='inner')
print(f"Merged: {len(monthly_df)} months")
monthly_df.head()

In [ ]:
import statsmodels.api as sm

TRANSACTION_COST_ANNUAL = 0.0
tc_monthly = TRANSACTION_COST_ANNUAL / 12.0

reg = monthly_df.merge(ff_all, on='YYYYMM', how='inner').dropna(
    subset=['Spread', 'MKT_RF', 'SMB', 'HML', 'UMD']
)
reg['Spread_Gross'] = reg['Spread']
reg['Spread'] = reg['Spread'] - tc_monthly
print(f"TC haircut: {TRANSACTION_COST_ANNUAL:.2%}/yr ({tc_monthly*100:.4f}%/mo) subtracted")

y = reg['Spread']

X_capm = sm.add_constant(reg[['MKT_RF']])
capm = sm.OLS(y, X_capm).fit()

X_4f = sm.add_constant(reg[['MKT_RF', 'SMB', 'HML', 'UMD']])
ff4 = sm.OLS(y, X_4f).fit()

raw_mean_ann = y.mean() * 12
raw_std_ann = y.std(ddof=1) * np.sqrt(12)
sharpe = raw_mean_ann / raw_std_ann

alpha_capm_m = capm.params['const']
alpha_4f_m = ff4.params['const']
resid_std_m = ff4.resid.std(ddof=1)
IR = (alpha_4f_m / resid_std_m) * np.sqrt(12)

print(f"\n=== RAW (gross of TC, monthly L/S, annualized) ===")
print(f"Return: {raw_mean_ann*100:.2f}%/yr,  Vol: {raw_std_ann*100:.2f}%/yr,  Sharpe: {sharpe:.2f}")

print(f"\n=== CAPM ===")
print(f"Alpha (annual): {alpha_capm_m*12*100:.2f}%")
print(f"Alpha t-stat:   {capm.tvalues['const']:.2f}")
print(f"Beta_MKT:       {capm.params['MKT_RF']:.2f}")
print(f"R^2:            {capm.rsquared:.3f}")

print(f"\n=== 4-FACTOR (Carhart) ===")
print(f"Alpha (annual): {alpha_4f_m*12*100:.2f}%")
print(f"Alpha t-stat:   {ff4.tvalues['const']:.2f}")
print(f"R^2:            {ff4.rsquared:.3f}")
print("Factor loadings:")
for f in ['MKT_RF', 'SMB', 'HML', 'UMD']:
    print(f"  {f:7s}: {ff4.params[f]:+.3f}  (t={ff4.tvalues[f]:+.2f})")

print(f"\nInformation Ratio (annualized): {IR:.2f}")


In [ ]:
#Out of sample predictions. 